# Session 8 · Case study: who gets a retention offer?

*Author: course team · Licence: CC-BY-4.0*

**The business question.** Each month the retention team of the telecom company may send an offer (for example a discount on the next contract) to customers who are likely to cancel. An offer costs money; a customer who leaves costs more. The churn model of Parts A–B ([05-case-study-churn-pipelines.ipynb](05-case-study-churn-pipelines.ipynb)) gives each customer a probability of churning. This workbook turns those probabilities into a list of names, and checks whether the probabilities can be trusted.

| Part | Block | Task |
|---|---|---|
| C | 2:00–2:45 | Choose the churn threshold from the cost of a retention offer against the value of a lost customer, test how sensitive it is to the cost assumptions, and check whether the predicted probabilities can be trusted |

**Assumptions** (to discuss with the business owner): an offer costs **€100**; a churner who receives it stays; a churner who receives none is lost, which costs **€400** in future margin. Theory page 03 derives the rule *offer if p > c / L*.

**Data.** The IBM Telco churn data (7,043 customers, 26.5 % churners), downloaded from IBM's GitHub repository. The notebook runs in about a minute. Tasks for you are marked **Task**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, make_scorer, roc_auc_score
from sklearn.model_selection import (StratifiedKFold, TunedThresholdClassifierCV, cross_val_predict,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # blanks -> NaN
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical = [c for c in X.columns if c not in numeric]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])
logreg = make_pipeline(pre, LogisticRegression(max_iter=1000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(len(X_train), "training and", len(X_test), "test customers")

## C1 · The cost of a campaign for every threshold

The cost of a campaign at threshold *t*: €100 for every flagged customer plus €400 for every churner who is not flagged. We compute it on **out-of-fold** probabilities of the training customers, so that the test set stays untouched until the end (Session 7).

In [ ]:
OFFER, LOST = 100, 400


def campaign_cost(y_true, proba, t, offer=OFFER, lost=LOST):
    flagged = proba >= t
    return offer * flagged.sum() + lost * (~flagged & (np.asarray(y_true) == 1)).sum()


oof = cross_val_predict(logreg, X_train, y_train, method="predict_proba", cv=cv)[:, 1]
ts = np.round(np.arange(0.02, 0.99, 0.01), 2)
costs = np.array([campaign_cost(y_train, oof, t) for t in ts])
t_best = ts[costs.argmin()]
print("best threshold", t_best, " rule c / L =", OFFER / LOST)
print("cost per customer: best", round(costs.min() / len(y_train), 1), " at 0.5", round(campaign_cost(y_train, oof, 0.5) / len(y_train), 1))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ts, costs / len(y_train))
ax.axvline(t_best, ls="--", color="grey")
ax.set_xlabel("threshold t"); ax.set_ylabel("cost per customer (€)")
ax.set_title("Out-of-fold cost of the campaign")
plt.show()

**Task C1.** (1) How many training customers would receive an offer at the best threshold, and what share of them are real churners (precision)? What share of all churners do they reach (recall)? (2) Is the curve flat or steep around the optimum? What does that mean for the discussion with the business owner?

In [ ]:
# your answer to C1 (1) here, for example:
# flagged = oof >= t_best; print(flagged.sum(), y_train[flagged].mean(), flagged[y_train == 1].mean())

## C2 · Let scikit-learn search the threshold, then test once

`TunedThresholdClassifierCV` does the same search with cross-validation for any scorer. The scorer must return a number where *higher is better*, so it returns minus the cost.

In [ ]:
def gain(y_true, y_pred):
    y_true = np.asarray(y_true)
    return -(OFFER * (y_pred == 1).sum() + LOST * ((y_true == 1) & (y_pred == 0)).sum())


tuned = TunedThresholdClassifierCV(logreg, scoring=make_scorer(gain), cv=5, random_state=0).fit(X_train, y_train)
print("cross-validated threshold:", round(tuned.best_threshold_, 3))

proba_test = logreg.fit(X_train, y_train).predict_proba(X_test)[:, 1]       # the one look at the test set
strategies = {"no offers": np.inf, "offers to everybody": 0.0, "default threshold 0.5": 0.5,
              "rule c / L": OFFER / LOST, "tuned threshold": tuned.best_threshold_}
pd.DataFrame([{"strategy": k, "threshold": t, "offers": int((proba_test >= t).sum()),
               "cost on test (€)": campaign_cost(y_test, proba_test, t)} for k, t in strategies.items()])

## C3 · How much do the cost assumptions matter?

Nobody knows the value of a lost customer exactly. We repeat the search for a range of values and for an offer that saves only some of the churners: if an offer keeps a churner with probability *s*, flagging a customer with churn probability *p* saves *s · L · p* and costs *c*, so the rule becomes *p > c / (s · L)*.

In [ ]:
rows = []
for lost in [200, 300, 400, 600, 800]:
    for success in [1.0, 0.5]:
        # a flagged churner is kept with probability `success`; the expected cost of a flagged churner is (1 - success) * lost
        c = [OFFER * (oof >= t).sum() + lost * ((oof < t) & (y_train == 1)).sum()
             + (1 - success) * lost * ((oof >= t) & (y_train == 1)).sum() for t in ts]
        rows.append({"lost customer €": lost, "offer success": success, "best threshold": ts[int(np.argmin(c))],
                     "rule c / (s L)": round(OFFER / (success * lost), 3),
                     "share flagged": round((oof >= ts[int(np.argmin(c))]).mean(), 3)})
pd.DataFrame(rows)

**Task C2–C3.** (1) How much does the tuned threshold save on the test customers compared with the default 0.5 and with doing nothing? (2) When the offer keeps only every second churner and a lost customer is worth €200, what does the table recommend? Explain the result to the retention team in two sentences. (3) Why must the threshold be chosen on the training data and only checked on the test set?

## C4 · Can the probabilities be trusted?

The rule *p > c / L* and any statement like "we expect 470 churners next month" assume that the probabilities are **calibrated**: among customers with p ≈ 0.3, about 30 % churn. We compare four models: the logistic regression, the same model trained with `class_weight="balanced"` (Session 9), 50-nearest neighbours and gradient boosting (Session 10; here a black box).

In [ ]:
candidates = {
    "logistic regression": make_pipeline(pre, LogisticRegression(max_iter=1000)),
    "logistic, class weights": make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced")),
    "50-NN": make_pipeline(pre, KNeighborsClassifier(n_neighbors=50)),
    "gradient boosting": make_pipeline(pre, HistGradientBoostingClassifier(random_state=0)),
}
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="perfectly calibrated")
rows = []
for name, m in candidates.items():
    q = m.fit(X_train, y_train).predict_proba(X_test)[:, 1]
    frac, mean_p = calibration_curve(y_test, q, n_bins=10, strategy="quantile")
    ax.plot(mean_p, frac, marker="o", label=name)
    rows.append({"model": name, "ROC AUC": roc_auc_score(y_test, q), "Brier": brier_score_loss(y_test, q),
                 "expected churners": q.sum(), "actual churners": y_test.sum(),
                 "cost with rule c / L": campaign_cost(y_test, q, OFFER / LOST)})
ax.set_xlabel("mean predicted probability (per bin)"); ax.set_ylabel("observed share of churners")
ax.legend(fontsize=8); plt.show()
pd.DataFrame(rows).round(3)

## C5 · Recalibrate the weighted model

`CalibratedClassifierCV` fits a mapping from the model's scores to calibrated probabilities on held-out folds of the training data (`method="sigmoid"`: Platt scaling).

In [ ]:
weighted_cal = CalibratedClassifierCV(candidates["logistic, class weights"], method="sigmoid", cv=5).fit(X_train, y_train)
q = weighted_cal.predict_proba(X_test)[:, 1]
print("recalibrated: AUC", round(roc_auc_score(y_test, q), 3), " Brier", round(brier_score_loss(y_test, q), 3),
      " expected churners", round(q.sum()), " cost with rule c / L", campaign_cost(y_test, q, OFFER / LOST))

tuned_w = TunedThresholdClassifierCV(candidates["logistic, class weights"], scoring=make_scorer(gain), cv=5,
                                     random_state=0).fit(X_train, y_train)
print("tuned threshold of the uncalibrated weighted model:", round(tuned_w.best_threshold_, 3))

In our run the weighted model ranked the customers as well as the unweighted one (ROC AUC 0.844 for both) but expected 725 churners among 1,761 test customers, of whom 467 churned; the rule *c / L* applied to its probabilities cost €116,400 instead of €109,200. After recalibration it expected 465 churners and the rule cost €108,700. A tuned threshold (about 0.52) compensates as well, because it looks at the costs, not at the probabilities.

**Task C4–C5.** (1) Which models are close to the diagonal, and which over- or under-estimate the churn risk? Does the ranking (ROC AUC) tell you that? (2) Finance wants a forecast of the number of churners next quarter from the sum of the probabilities. Which model would you give them, and why? (3) Write a short recommendation for the retention team (five sentences): model, threshold, expected cost per customer on the test set, the assumptions behind the costs, and what should be checked again in three months.

**Team project.** If your project ends in a yes/no decision, write down what each kind of error costs and who decided these costs. If you report probabilities, add a calibration curve.